## 1. Import packages

Import necessary packages required such as Pandas and Docplex.

In [1]:
import pandas as pd
from docplex.mp.model import Model

## 2. Load data

Load data as 3 separate dataframes:

1. `new_students` : new students' ID, tutoring need, and tuition centre
2. `tutors` : tutors' ID, tutoring skills, preferred centres (1 and 2), maximum overall capacity for number of students
3. `existing_students` : existing students' ID, tutor ID, tutoring need, tuition centre and activity status

In [2]:
filepath = "Interview small data.xlsx"
new_students = pd.read_excel(filepath, sheet_name = "New Students")
tutors = pd.read_excel(filepath, sheet_name = "Tutor Information")
existing_students = pd.read_excel(filepath, sheet_name = "Existing Students")

#For results at the end
output_file = "HTX_assignment_results.xlsx"

In [3]:
new_students.head()

,studentId,tutoringNeed,tuitionCentre
0,S0001,Normal,East
1,S0002,Normal,West
2,S0003,Normal,Central
3,S0004,Normal,East
4,S0005,Normal,Central


In [4]:
tutors.head()

,tutorId,tutoringSkills,preferredCentre1,preferredCentre2,maxOverallCapacity
0,A001,Extensive,East,North,8
1,A002,Normal,North,East,5
2,A003,Normal,West,North,5
3,A004,Normal,North,East,3
4,A005,Normal,North,Central,6


In [5]:
existing_students.head()

,tutorId,studentId,tutoringNeed,tuitionCentre,active
0,A001,S0021,Normal,East,False
1,A001,S0022,Normal,North,False
2,A001,S0023,Extensive,East,False
3,A001,S0024,Extensive,East,False
4,A001,S0025,Normal,North,False


## 3. Data preprocessing

We will calculate the remaining capacity per tutor to figure out how many new students the tutors can manage. This will be done by first filtering out inactive students and calculating how many existing students each tutor has.

In [6]:
active_students = existing_students[existing_students["active"]]
existing_count = active_students.groupby("tutorId").size().reset_index(name="existingCount")
existing_count

,tutorId,existingCount
0,A003,1
1,A005,4
2,A007,5
3,A010,2


In [7]:
tutors = tutors.merge(existing_count, on="tutorId", how="left")
tutors["existingCount"] = tutors["existingCount"].fillna(0).astype(int)
tutors


,tutorId,tutoringSkills,preferredCentre1,preferredCentre2,maxOverallCapacity,existingCount
0,A001,Extensive,East,North,8,0
1,A002,Normal,North,East,5,0
2,A003,Normal,West,North,5,1
3,A004,Normal,North,East,3,0
4,A005,Normal,North,Central,6,4
5,A006,Extensive,East,Central,5,0
6,A007,Normal,East,North,8,5
7,A008,Normal,Central,West,5,0
8,A009,Extensive,Central,North,4,0
9,A010,Normal,West,North,8,2


We will then calculate the remaining capacity of each tutor by taking the difference from maximum overall capacity and existing count of students. This will be appended as a new column `remainingCapacity`

In [8]:
tutors["remainingCapacity"] = tutors["maxOverallCapacity"] - tutors["existingCount"]
tutors

,tutorId,tutoringSkills,preferredCentre1,preferredCentre2,maxOverallCapacity,existingCount,remainingCapacity
0,A001,Extensive,East,North,8,0,8
1,A002,Normal,North,East,5,0,5
2,A003,Normal,West,North,5,1,4
3,A004,Normal,North,East,3,0,3
4,A005,Normal,North,Central,6,4,2
5,A006,Extensive,East,Central,5,0,5
6,A007,Normal,East,North,8,5,3
7,A008,Normal,Central,West,5,0,5
8,A009,Extensive,Central,North,4,0,4
9,A010,Normal,West,North,8,2,6


## 4. Model formulation

The objective of this optimisation model is to assign new students to tutors while satisfying constraints and priorities.

The model must ensure:

- Every student is assigned exactly one tutor.
- Tutors are only assigned students they are qualified to teach.
- Tutors are not overloaded beyond their maximum capacity.

We implemented the model using IBM CPLEX through the `docplex` Python library, which allows us to define decision variables, constraints, and objective functions programmatically.

### 4.1 Decision variables

The core decision variable in this model is:

- **x[s, t]**: A binary decision variable that equals 1 if student `s` is assigned to tutor `t`, and 0 otherwise.

This allows the model to determine the optimal assignment configuration across all students and tutors.

When different problems are addressed (minimising tutors used, balancing workload, etc.), additional  variables will be added.

These variables are defined using `docplex.binary_var_matrix()` and `docplex.binary_var_dict()` respectively.

In [9]:
mdl = Model("TutorAssignment")

student_ids = new_students["studentId"].tolist()
tutor_ids = tutors["tutorId"].tolist()

x = mdl.binary_var_matrix(student_ids, tutor_ids, name="x")

### 4.2 Constraints

Constraint 1: Each student must be assigned to one tutor

In [10]:
for s in student_ids:
    mdl.add_constraint(mdl.sum(x[s, t] for t in tutor_ids) == 1)

Constraint 2: Students' requiring extensive teaching must receive tutors with extensive skill

If a tutor does not possess the required skill level, the variable is forced to zero.

In [11]:
for _, s_row in new_students.iterrows():
    for _, t_row in tutors.iterrows():
        if s_row["tutoringNeed"] == "Extensive" and t_row["tutoringSkills"] != "Extensive":
            mdl.add_constraint(x[s_row["studentId"], t_row["tutorId"]] == 0)

Constraint 3: Tutor must not exceed maximum capacity

Existing students + newly assigned students ≤ Maximum capacity

This is ensured by:

newly assigned students ≤ remaining capacity

In [12]:
y = mdl.binary_var_dict(tutor_ids, name="y")

for _, t_row in tutors.iterrows():
    t_id = t_row["tutorId"]
    remaining = t_row["remainingCapacity"]
    mdl.add_constraint(mdl.sum(x[s, t_id] for s in student_ids) <= remaining * y[t_id])

### 4.3 Preference score

To align with tutors' location preferences, we define a preference scoring system:

- 2 points if assigned to first-choice centre
- 1 point if assigned to second-choice centre
- 0 points otherwise

The total preference score is calculated across all assignments and incorporated into the objective function.

Preference alignment is encouraged, but not enforced as a strict constraint.

In [13]:
preference_score = {}

for _, s_row in new_students.iterrows():
    s = s_row["studentId"]
    centre = s_row["tuitionCentre"]

    for _, t_row in tutors.iterrows():
        t = t_row["tutorId"]
        pref1 = t_row["preferredCentre1"]
        pref2 = t_row["preferredCentre2"]

        if centre == pref1:
            score = 2
        elif centre == pref2:
            score = 1
        else:
            score = 0

        preference_score[(s, t)] = score

# 5. Solving for Scenario (a)

Minimizing number of tutors while maximizing the preference score.

### 5.1 Objective function

A weighted objective function is used, where tutor minimisation is assigned higher priority than preference satisfaction.

α = 100  
β = 1

minimizing: (α * number of tutors) - (β * preference score)



In [14]:
a = 100
b = 1

mdl.minimize(
    a * mdl.sum(y[t] for t in tutor_ids)- b * mdl.sum(
        preference_score[(s, t)] * x[s, t]
        for s in student_ids
        for t in tutor_ids
    )
)

### 5.2 Solving the model

In [15]:
solution_a = mdl.solve(log_output=True)

print("Status:", solution_a.solve_status)

Version identifier: 22.1.2.0 | 2024-12-09 | 8bd2200c8
CPXPARAM_Read_DataCheck                          1
Tried aggregator 1 time.
MIP Presolve eliminated 14 rows and 14 columns.
Reduced MIP has 30 rows, 196 columns, and 382 nonzeros.
Reduced MIP has 196 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.31 ticks)
Found incumbent of value 982.000000 after 0.00 sec. (0.79 ticks)
Probing time = 0.00 sec. (0.18 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 30 rows, 196 columns, and 382 nonzeros.
Reduced MIP has 196 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.39 ticks)
Probing time = 0.00 sec. (0.18 ticks)
Clique table members: 206.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 8 threads.
Root relaxation solution time = 0.00 sec. (0.16 ticks)

        Nodes                                         Cuts/
   Node  Left     Objecti

Model has been successfully solved.

In [16]:
assignments_a = []
for s in student_ids:
    for t in tutor_ids:
        if x[s, t].solution_value > 0.5:
            s_row = new_students.loc[new_students["studentId"] == s].iloc[0]
            t_row = tutors.loc[tutors["tutorId"] == t].iloc[0]
            assignments_a.append({
                "studentId": s,
                "tutorId": t,
                "studentNeed": s_row["tutoringNeed"],
                "tuitionCentre": s_row["tuitionCentre"],
                "tutorSkill": t_row["tutoringSkills"],
                "pref1": t_row["preferredCentre1"],
                "pref2": t_row["preferredCentre2"],
                "prefScore": preference_score[(s, t)]
            })

assign_a_df = pd.DataFrame(assignments_a)
assign_a_df

,studentId,tutorId,studentNeed,tuitionCentre,tutorSkill,pref1,pref2,prefScore
0,S0001,A001,Normal,East,Extensive,East,North,2
1,S0002,A010,Normal,West,Normal,West,North,2
2,S0003,A009,Normal,Central,Extensive,Central,North,2
3,S0004,A001,Normal,East,Extensive,East,North,2
4,S0005,A009,Normal,Central,Extensive,Central,North,2
5,S0006,A001,Extensive,North,Extensive,East,North,1
6,S0007,A001,Normal,East,Extensive,East,North,2
7,S0008,A009,Normal,Central,Extensive,Central,North,2
8,S0009,A002,Normal,North,Normal,North,East,2
9,S0010,A010,Normal,West,Normal,West,North,2


The new assignments for each tutor and student are as follows:

In [17]:
for s in student_ids:
    for t in tutor_ids:
        if x[s, t].solution_value > 0.5:
            print(f"Student {s} -> Tutor {t}")

Student S0001 -> Tutor A001
Student S0002 -> Tutor A010
Student S0003 -> Tutor A009
Student S0004 -> Tutor A001
Student S0005 -> Tutor A009
Student S0006 -> Tutor A001
Student S0007 -> Tutor A001
Student S0008 -> Tutor A009
Student S0009 -> Tutor A002
Student S0010 -> Tutor A010
Student S0011 -> Tutor A009
Student S0012 -> Tutor A010
Student S0013 -> Tutor A010
Student S0014 -> Tutor A002
Student S0015 -> Tutor A001
Student S0016 -> Tutor A002
Student S0017 -> Tutor A002
Student S0018 -> Tutor A010
Student S0019 -> Tutor A010
Student S0020 -> Tutor A002


The new workload for each tutor is reflected in the `afterAssignment` column.

In [18]:
new_assigned_a = (
    assign_a_df.groupby("tutorId")
    .size()
    .reset_index(name="newAssigned"))

tutor_summary_a = tutors[["tutorId", "maxOverallCapacity", "existingCount"]].merge(
    new_assigned_a, on="tutorId", how="left")

tutor_summary_a["newAssigned"] = tutor_summary_a["newAssigned"].fillna(0).astype(int)
tutor_summary_a["afterAssignment"] = tutor_summary_a["existingCount"] + tutor_summary_a["newAssigned"]

tutor_summary_a

,tutorId,maxOverallCapacity,existingCount,newAssigned,afterAssignment
0,A001,8,0,5,5
1,A002,5,0,5,5
2,A003,5,1,0,1
3,A004,3,0,0,0
4,A005,6,4,0,4
5,A006,5,0,0,0
6,A007,8,5,0,5
7,A008,5,0,0,0
8,A009,4,0,4,4
9,A010,8,2,6,8


### 7.3 Key metrics

Key metric 1: Number of tutors used

In [19]:
tutors_used = [t for t in tutor_ids if y[t].solution_value > 0.5]
num_tutors_used = len(tutors_used)
print(f"Number of tutors used: {num_tutors_used} out of {len(tutors)} tutors")
tutors_used

Number of tutors used: 4 out of 10 tutors


['A001', 'A002', 'A009', 'A010']

Key metric 2: Preference satisfaction rate

Percentage of assignments that matched either first or second choice.


In [20]:
pref1_count = (assign_a_df["prefScore"] == 2).sum()
pref2_count = (assign_a_df["prefScore"] == 1).sum()
other_count = (assign_a_df["prefScore"] == 0).sum()

total_assignments = len(assign_a_df)

success_rate = (pref1_count + pref2_count) / total_assignments * 100

print(f"First-choice assignments: {pref1_count}")
print(f"Second-choice assignments: {pref2_count}")
print(f"Other-centre assignments: {other_count}")
print(f"Total assignments: {total_assignments}")
print("Overall preference satisfaction rate (%):", round(success_rate, 2))

First-choice assignments: 19
Second-choice assignments: 1
Other-centre assignments: 0
Total assignments: 20
Overall preference satisfaction rate (%): 100.0


# 6. Solving for Scenario (b)

We will now balance workload instead of minimizing tutors and continue to maximize the preference score.

In [21]:
mdl.remove_objective()

### 6.1 Objective function

In Scenario (b), the objective shifts from minimising the number of tutors used to balancing workload across tutors.

Workload is defined as the total number of students a tutor has after assignment:

afterAssignment = existing students + newly assigned students.

To achieve workload balance, we minimise the spread between:

- The tutor with the highest total student count (`max_after`), and
- The tutor with the lowest total student count (`min_after`)

In [22]:
max_after = mdl.integer_var(lb=0, name="max_after")
min_after = mdl.integer_var(lb=0, name="min_after")

existing_map = dict(zip(tutors["tutorId"], tutors["existingCount"]))

for t in tutor_ids:
    total_load_t = existing_map[t] + mdl.sum(x[s, t] for s in student_ids)
    mdl.add_constraint(total_load_t <= max_after)
    mdl.add_constraint(total_load_t >= min_after)

As in (a), tutor preference alignment is still secondary, and is incorporated through the preference scoring system.

A weighted objective function is used, where workload balance is assigned higher priority than preference satisfaction.

γ = 100  
β = 1  

Minimizing:

(γ × workload spread) − (β × preference score)

Where workload spread = (maximum after-assignment load − minimum after-assignment load)

In [23]:
gamma = 100
b = 1

mdl.minimize(
    gamma * (max_after - min_after)
    - b * mdl.sum(preference_score[(s, t)] * x[s, t]
                     for s in student_ids for t in tutor_ids)
)

### 6.2 Solving the model

In [24]:
solution_b = mdl.solve(log_output=True)
print("Status:", solution_b.solve_status)

Version identifier: 22.1.2.0 | 2024-12-09 | 8bd2200c8
CPXPARAM_Read_DataCheck                          1
1 of 5 MIP starts provided solutions.
MIP start 'm1' defined initial solution with objective 761.0000.
Tried aggregator 1 time.
MIP Presolve eliminated 22 rows and 24 columns.
Reduced MIP has 42 rows, 188 columns, and 608 nonzeros.
Reduced MIP has 186 binaries, 2 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.66 ticks)
Probing time = 0.00 sec. (0.20 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 42 rows, 188 columns, and 608 nonzeros.
Reduced MIP has 186 binaries, 2 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.50 ticks)
Probing time = 0.00 sec. (0.20 ticks)
Clique table members: 20.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 8 threads.
Root relaxation solution time = 0.00 sec. (0.19 ticks)

        Nodes                                   

Model has been successfully solved.

In [25]:
assignments_b = []
for s in student_ids:
    for t in tutor_ids:
        if x[s, t].solution_value > 0.5:
            s_row = new_students.loc[new_students["studentId"] == s].iloc[0]
            t_row = tutors.loc[tutors["tutorId"] == t].iloc[0]
            assignments_b.append({
                "studentId": s,
                "tutorId": t,
                "studentNeed": s_row["tutoringNeed"],
                "tuitionCentre": s_row["tuitionCentre"],
                "tutorSkill": t_row["tutoringSkills"],
                "pref1": t_row["preferredCentre1"],
                "pref2": t_row["preferredCentre2"],
                "prefScore": preference_score[(s, t)]
            })

assign_b_df = pd.DataFrame(assignments_b)
assign_b_df

,studentId,tutorId,studentNeed,tuitionCentre,tutorSkill,pref1,pref2,prefScore
0,S0001,A006,Normal,East,Extensive,East,Central,2
1,S0002,A003,Normal,West,Normal,West,North,2
2,S0003,A008,Normal,Central,Normal,Central,West,2
3,S0004,A006,Normal,East,Extensive,East,Central,2
4,S0005,A009,Normal,Central,Extensive,Central,North,2
5,S0006,A001,Extensive,North,Extensive,East,North,1
6,S0007,A001,Normal,East,Extensive,East,North,2
7,S0008,A008,Normal,Central,Normal,Central,West,2
8,S0009,A002,Normal,North,Normal,North,East,2
9,S0010,A003,Normal,West,Normal,West,North,2


The new assignments for each tutor and student are as follows:

In [26]:
for s in student_ids:
    for t in tutor_ids:
        if x[s, t].solution_value > 0.5:
            print(f"Student {s} -> Tutor {t}")

Student S0001 -> Tutor A006
Student S0002 -> Tutor A003
Student S0003 -> Tutor A008
Student S0004 -> Tutor A006
Student S0005 -> Tutor A009
Student S0006 -> Tutor A001
Student S0007 -> Tutor A001
Student S0008 -> Tutor A008
Student S0009 -> Tutor A002
Student S0010 -> Tutor A003
Student S0011 -> Tutor A009
Student S0012 -> Tutor A010
Student S0013 -> Tutor A010
Student S0014 -> Tutor A002
Student S0015 -> Tutor A001
Student S0016 -> Tutor A002
Student S0017 -> Tutor A004
Student S0018 -> Tutor A003
Student S0019 -> Tutor A010
Student S0020 -> Tutor A004


The new workload for each tutor is reflected in the `afterAssignment` column.

In [27]:
new_assigned_b = (
    assign_b_df.groupby("tutorId")
    .size()
    .reset_index(name="newAssigned"))

tutor_summary_b = tutors[["tutorId", "maxOverallCapacity", "existingCount"]].merge(
    new_assigned_b, on="tutorId", how="left")

tutor_summary_b["newAssigned"] = tutor_summary_b["newAssigned"].fillna(0).astype(int)
tutor_summary_b["afterAssignment"] = tutor_summary_b["existingCount"] + tutor_summary_b["newAssigned"]
    
tutor_summary_b

,tutorId,maxOverallCapacity,existingCount,newAssigned,afterAssignment
0,A001,8,0,3,3
1,A002,5,0,3,3
2,A003,5,1,3,4
3,A004,3,0,2,2
4,A005,6,4,0,4
5,A006,5,0,2,2
6,A007,8,5,0,5
7,A008,5,0,2,2
8,A009,4,0,2,2
9,A010,8,2,3,5


### 6.3 Key metrics

Key metric 1: Minimizing spread (balancing workload)

In [28]:
print("AfterAssignment min:", tutor_summary_b["afterAssignment"].min())
print("AfterAssignment max:", tutor_summary_b["afterAssignment"].max())
print("Spread:", tutor_summary_b["afterAssignment"].max() - tutor_summary_b["afterAssignment"].min())

AfterAssignment min: 2
AfterAssignment max: 5
Spread: 3


Key metric 2: Preference satisfaction rate

Percentage of assignments that matched either first or second choice.

In [29]:
pref1_count = (assign_b_df["prefScore"] == 2).sum()
pref2_count = (assign_b_df["prefScore"] == 1).sum()
other_count = (assign_b_df["prefScore"] == 0).sum()

total_assignments = len(assign_b_df)
success_rate = (pref1_count + pref2_count) / total_assignments * 100

print(f"First-choice assignments: {pref1_count}")
print(f"Second-choice assignments: {pref2_count}")
print(f"Other-centre assignments: {other_count}")
print(f"Total assignments: {total_assignments}")
print("Overall preference satisfaction rate (%):", round(success_rate, 2))

First-choice assignments: 19
Second-choice assignments: 1
Other-centre assignments: 0
Total assignments: 20
Overall preference satisfaction rate (%): 100.0


# 7. Results

### 7.1 Scenario (a)

Scenario (a) prioritises minimising the number of tutors activated.

Results:
- Tutors used: 4
- Workload range (after assignment): 0 to 8 students
- Workload spread: 8 students
- Preference satisfaction rate: 100%

This configuration concentrates students among fewer tutors, improving operational efficiency.
However, workload is less evenly distributed, with some tutors operating at maximum capacity while others have no students.

### 7.2 Scenario (b)

Scenario (b) prioritises minimising workload spread.

Results:
- Tutors used: 10
- Workload range (after assignment): 2 to 5 students
- Workload spread: 3 students
- Preference satisfaction rate: 100%

Compared to Scenario (a), Scenario (b) produces a more balanced distribution of students across tutors.
The drawback would be manpower and operational costs as 6 extra tutors (and by extension 6 extra lessons) are required.

Preference satisfaction remains strong despite shifting priority toward fairness.

### 7.3 Trade-Off Discussion

The comparison highlights a clear trade-off:

- Scenario (a) improves operational efficiency by activating fewer tutors.
- Scenario (b) improves fairness by reducing workload imbalance.

While Scenario (a) may reduce administrative overhead, it risks uneven workload concentration.
Scenario (b) promotes more equitable and sustainable allocation patterns.




But what if we could achieve the best of both worlds by combining the objectives into a new model?

# 8. Solve for Scenario (c): Combined Objective

Scenario (c) integrates the objectives of Scenario (a) and Scenario (b).

Instead of prioritising only efficiency or only fairness, this scenario balances:

- Minimising number of tutors used (operational efficiency)
- Minimising workload spread (fairness)
- Maximising tutor preference satisfaction

In [30]:
mdl.remove_objective()

### 8.1 Objective function

A combined weighted objective is used:

α = 50  
γ = 50  
β = 1  

Minimising:

(α × number of tutors used) + (γ × workload spread) − (β × preference score)

Where:
- α controls efficiency importance
- γ controls workload fairness importance
- β controls preference alignment

By assigning similar magnitudes to α and γ, the model balances both efficiency and fairness simultaneously.

In [31]:
alpha = 50
gamma = 50
beta  = 1

mdl.minimize(
    alpha * mdl.sum(y[t] for t in tutor_ids)
    + gamma * (max_after - min_after)
    - beta  * mdl.sum(preference_score[(s, t)] * x[s, t]
                      for s in student_ids for t in tutor_ids)
)

### 8.2 Solving the model

In [32]:
solution_c = mdl.solve(log_output=True)
print("Status:", solution_c.solve_status)

Version identifier: 22.1.2.0 | 2024-12-09 | 8bd2200c8
CPXPARAM_Read_DataCheck                          1
3 of 3 MIP starts provided solutions.
MIP start 'm3' defined initial solution with objective 561.0000.
Tried aggregator 1 time.
MIP Presolve eliminated 22 rows and 14 columns.
Reduced MIP has 42 rows, 198 columns, and 618 nonzeros.
Reduced MIP has 196 binaries, 2 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.70 ticks)
Probing time = 0.00 sec. (0.24 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 42 rows, 198 columns, and 618 nonzeros.
Reduced MIP has 196 binaries, 2 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.54 ticks)
Probing time = 0.00 sec. (0.24 ticks)
Clique table members: 206.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 8 threads.
Root relaxation solution time = 0.00 sec. (0.40 ticks)

        Nodes                                  

Model has been successfully solved.

In [33]:
assignments_c = []
for s in student_ids:
    for t in tutor_ids:
        if x[s, t].solution_value > 0.5:
            s_row = new_students.loc[new_students["studentId"] == s].iloc[0]
            t_row = tutors.loc[tutors["tutorId"] == t].iloc[0]
            assignments_c.append({
                "studentId": s,
                "tutorId": t,
                "studentNeed": s_row["tutoringNeed"],
                "tuitionCentre": s_row["tuitionCentre"],
                "tutorSkill": t_row["tutoringSkills"],
                "pref1": t_row["preferredCentre1"],
                "pref2": t_row["preferredCentre2"],
                "prefScore": preference_score[(s, t)]
            })

assign_c_df = pd.DataFrame(assignments_c)
assign_c_df

,studentId,tutorId,studentNeed,tuitionCentre,tutorSkill,pref1,pref2,prefScore
0,S0001,A001,Normal,East,Extensive,East,North,2
1,S0002,A008,Normal,West,Normal,Central,West,1
2,S0003,A006,Normal,Central,Extensive,East,Central,1
3,S0004,A001,Normal,East,Extensive,East,North,2
4,S0005,A006,Normal,Central,Extensive,East,Central,1
5,S0006,A001,Extensive,North,Extensive,East,North,1
6,S0007,A001,Normal,East,Extensive,East,North,2
7,S0008,A008,Normal,Central,Normal,Central,West,2
8,S0009,A002,Normal,North,Normal,North,East,2
9,S0010,A006,Normal,West,Extensive,East,Central,0


The new assignments for each tutor and student are as follows:

In [34]:
for s in student_ids:
    for t in tutor_ids:
        if x[s, t].solution_value > 0.5:
            print(f"Student {s} -> Tutor {t}")

Student S0001 -> Tutor A001
Student S0002 -> Tutor A008
Student S0003 -> Tutor A006
Student S0004 -> Tutor A001
Student S0005 -> Tutor A006
Student S0006 -> Tutor A001
Student S0007 -> Tutor A001
Student S0008 -> Tutor A008
Student S0009 -> Tutor A002
Student S0010 -> Tutor A006
Student S0011 -> Tutor A006
Student S0012 -> Tutor A008
Student S0013 -> Tutor A006
Student S0014 -> Tutor A002
Student S0015 -> Tutor A001
Student S0016 -> Tutor A002
Student S0017 -> Tutor A002
Student S0018 -> Tutor A008
Student S0019 -> Tutor A008
Student S0020 -> Tutor A002


The new workload for each tutor is reflected in the `afterAssignment` column.

In [35]:
new_assigned_c = (
    assign_c_df.groupby("tutorId")
    .size()
    .reset_index(name="newAssigned"))

tutor_summary_c = tutors[["tutorId", "maxOverallCapacity", "existingCount"]].merge(
    new_assigned_c, on="tutorId", how="left")

tutor_summary_c["newAssigned"] = tutor_summary_c["newAssigned"].fillna(0).astype(int)
tutor_summary_c["afterAssignment"] = tutor_summary_c["existingCount"] + tutor_summary_c["newAssigned"]

tutor_summary_c

,tutorId,maxOverallCapacity,existingCount,newAssigned,afterAssignment
0,A001,8,0,5,5
1,A002,5,0,5,5
2,A003,5,1,0,1
3,A004,3,0,0,0
4,A005,6,4,0,4
5,A006,5,0,5,5
6,A007,8,5,0,5
7,A008,5,0,5,5
8,A009,4,0,0,0
9,A010,8,2,0,2


### 8.3 Key Metrics

In [36]:
#Tutors used
tutors_used_c = [t for t in tutor_ids if y[t].solution_value > 0.5]
print(f"- Tutors used: {tutors_used_c}")

#Workload range
min_after_c = tutor_summary_c["afterAssignment"].min()
max_after_c = tutor_summary_c["afterAssignment"].max()
print(f"- Workload range (after assignment): {min_after_c} to {max_after_c} students")

#Workload spread
spread_c = max_after_c - min_after_c
print(f"- Workload spread: {spread_c} students")

#Preference satisfaction rate
pref1_c = (assign_c_df["prefScore"] == 2).sum()
pref2_c = (assign_c_df["prefScore"] == 1).sum()
total_assignments_c = len(assign_c_df)
success_rate_c = (pref1_c + pref2_c) / total_assignments_c * 100
print(f"- Preference satisfaction rate: {round(success_rate_c, 2)}%")

- Tutors used: ['A001', 'A002', 'A006', 'A008']
- Workload range (after assignment): 0 to 5 students
- Workload spread: 5 students
- Preference satisfaction rate: 90.0%


# 9. Conclusion

Three strategic scenarios were evaluated:

- Scenario (a) prioritised operational efficiency by minimising the number of tutors used.
- Scenario (b) prioritised fairness by minimising workload spread.
- Scenario (c) combined both objectives to produce a balanced compromise solution.

Under Scenario (c), only 4 tutors were activated while maintaining a workload range of 2 to 5 students and achieving a 90% preference satisfaction rate. This highlights the trade-off between efficiency, fairness, and preference alignment.

Overall, the model provides a flexible and scalable decision-support tool that can be adapted to evolving operational priorities while maintaining transparency and consistency in assignment decisions.



The output spreadsheet containing all the student assignments and tutors information for all 3 scenarios can be found below.

In [37]:
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    #scenario (a)
    assign_a_df.to_excel(writer, sheet_name="(a)_student_assignments", index=False)
    tutor_summary_a.to_excel(writer, sheet_name="(a)_tutors", index=False)

    #scenario (b)
    assign_b_df.to_excel(writer, sheet_name="(b)_student_assignments", index=False)
    tutor_summary_b.to_excel(writer, sheet_name="(b)_tutors", index=False)

    #scenario (c)
    assign_c_df.to_excel(writer, sheet_name="(c)_student_assignments", index=False)
    tutor_summary_c.to_excel(writer, sheet_name="(c)_tutors", index=False)

print(f"Excel file successfully created: {output_file}")

Excel file successfully created: HTX_assignment_results.xlsx
